# SmolVLA LIBERO-Object Evaluation

**Checkpoint**: `dennywu2966/smolvla-libero-object-lora` (20k steps, loss ~0.134)

**Key facts** (verified locally):
- `config.json` already has `use_peft=True` — no extra CLI flag needed
- Normalizer stats are 8D (matches LiberoProcessorStep eef_pos+axisangle+gripper)
- rename_map (image→camera1, image2→camera2) is embedded in `policy_preprocessor.json`
- `LIBERO_DATA_FOLDER` set before import to skip interactive prompt

In [ ]:
# Cell 0: Install — explicit versions matching training setup
!pip install -q "lerobot[smolvla,peft,libero]>=0.4.3" "peft>=0.18.0"

import os, subprocess, sys

# CRITICAL: set LIBERO_DATA_FOLDER BEFORE importing libero to skip interactive prompt
os.makedirs("/tmp/libero_data", exist_ok=True)
os.environ["LIBERO_DATA_FOLDER"] = "/tmp/libero_data"
os.environ["MUJOCO_GL"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Verify LIBERO is installed (import after setting env var)
try:
    import libero  # noqa
    print("LIBERO: OK")
except ImportError as e:
    print(f"LIBERO missing: {e}")
    raise

In [ ]:
# Cell 1: Verify checkpoint integrity
from huggingface_hub import hf_hub_download
from peft import PeftConfig
import json

LORA_REPO = "dennywu2966/smolvla-libero-object-lora"

peft_cfg = PeftConfig.from_pretrained(LORA_REPO)
print(f"base_model: {peft_cfg.base_model_name_or_path}")
print(f"peft_type: {peft_cfg.peft_type}, r={peft_cfg.r}")
assert peft_cfg.base_model_name_or_path == "lerobot/smolvla_base", "Wrong base model!"

cfg_path = hf_hub_download(LORA_REPO, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
print(f"use_peft in config.json: {cfg.get('use_peft')}  (should be True)")
assert cfg.get("use_peft") is True, "use_peft not True in config.json!"

print("Checkpoint OK — ready for eval")

In [ ]:
# Cell 2: Evaluate SmolVLA-LoRA fine-tuned
# use_peft=True is already in config.json — read automatically by lerobot-eval
import subprocess, time, os

LORA_REPO = "dennywu2966/smolvla-libero-object-lora"

# LIBERO_DATA_FOLDER must be set to skip interactive prompt in subprocess
eval_env = {
    **os.environ,
    "MUJOCO_GL": "egl",
    "TOKENIZERS_PARALLELISM": "false",
    "LIBERO_DATA_FOLDER": "/tmp/libero_data",
}

print(f"Evaluating: {LORA_REPO}")
print("Expected: ~2-3 hours for 20 episodes x 10 tasks")

start = time.time()
proc = subprocess.Popen(
    [
        "lerobot-eval",
        f"--policy.path={LORA_REPO}",
        "--policy.device=cuda",
        "--env.type=libero",
        "--env.task=libero_object",
        "--eval.n_episodes=20",
        "--eval.batch_size=1",
    ],
    env=eval_env,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Feed "N" to answer LIBERO's "specify custom data folder? (Y/N)" prompt
proc.stdin.write("N\n")
proc.stdin.flush()
proc.stdin.close()

ft_output = []
for line in proc.stdout:
    print(line, end="", flush=True)
    ft_output.append(line)

proc.wait(timeout=14400)  # 4h hard timeout
elapsed = time.time() - start
print(f"\nExit: {proc.returncode} | Time: {elapsed/3600:.2f}h")

if proc.returncode != 0:
    print("\n=== EVAL FAILED — last 30 lines ===")
    print("".join(ft_output[-30:]))

In [ ]:
# Cell 3: Evaluate official HuggingFaceVLA/smolvla_libero (upper bound reference)
import subprocess, time, os

OFFICIAL_REPO = "HuggingFaceVLA/smolvla_libero"
eval_env = {
    **os.environ,
    "MUJOCO_GL": "egl",
    "TOKENIZERS_PARALLELISM": "false",
    "LIBERO_DATA_FOLDER": "/tmp/libero_data",
}

print(f"Evaluating official reference: {OFFICIAL_REPO}")

start = time.time()
proc = subprocess.Popen(
    [
        "lerobot-eval",
        f"--policy.path={OFFICIAL_REPO}",
        "--policy.device=cuda",
        "--env.type=libero",
        "--env.task=libero_object",
        "--eval.n_episodes=20",
        "--eval.batch_size=1",
    ],
    env=eval_env,
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Feed "N" to answer LIBERO's "specify custom data folder? (Y/N)" prompt
proc.stdin.write("N\n")
proc.stdin.flush()
proc.stdin.close()

official_output = []
for line in proc.stdout:
    print(line, end="", flush=True)
    official_output.append(line)

proc.wait(timeout=14400)
elapsed = time.time() - start
print(f"\nExit: {proc.returncode} | Time: {elapsed/3600:.2f}h")

if proc.returncode != 0:
    print("\n=== EVAL FAILED — last 30 lines ===")
    print("".join(official_output[-30:]))

In [ ]:
# Cell 4: Parse results from outputs/ directory
import glob, json, re

def parse_eval_results(output_lines, label):
    """Parse success rates from lerobot-eval stdout."""
    for line in output_lines:
        m = re.search(r"(pc_success|avg_sum_reward|success).*?([\d.]+)", line)
        if m:
            print(f"[{label}] {line.rstrip()}")
    files = sorted(glob.glob("outputs/eval/*/eval_info.json"))
    print(f"\nResult files: {files}")
    results = {}
    for f in files:
        with open(f) as fh:
            data = json.load(fh)
        print(f"\n--- {f} ---")
        overall = data.get("overall", {})
        print(f"pc_success: {overall.get('pc_success', 'N/A')}")
        print(f"avg_sum_reward: {overall.get('avg_sum_reward', 'N/A')}")
        results[f] = data
    return results

print("=== SmolVLA-LoRA results ===")
ft_results = parse_eval_results(ft_output, "FT")

print("\n=== Official SmolVLA results ===")
official_results = parse_eval_results(official_output, "Official")

In [ ]:
# Cell 5: Final comparison table (no external deps)
import json, glob

result_files = sorted(glob.glob("outputs/eval/*/eval_info.json"))
print(f"Found {len(result_files)} result files")

summaries = []
for f in result_files:
    with open(f) as fh:
        data = json.load(fh)
    overall = data.get("overall", {})
    policy_path = data.get("policy_path", f)
    pc = overall.get("pc_success", None)
    summaries.append({"policy": policy_path, "pc_success": pc, "file": f})
    print(f"{policy_path}: pc_success={pc}")

# Reference baselines
print("\n=== Comparison ===")
print(f"{'Model':<45} {'Success':>8}")
print("-" * 55)
for s in summaries:
    label = s["policy"].split("/")[-1] if "/" in str(s["policy"]) else str(s["policy"])
    pct = f"{s['pc_success']*100:.1f}%" if s["pc_success"] is not None else "N/A"
    print(f"{label:<45} {pct:>8}")
print(f"{'SmolVLA-base (zero-shot)':<45} {'0.0%':>8}")
print(f"{'OpenVLA-7B-FT (reference)':<45} {'88.4%':>8}")
print(f"{'SmolVLA-official-FT (reference)':<45} {'~90%+':>8}")

In [ ]:
# Cell 6: Save results artifact
import glob, json, shutil

all_results = {}
for f in sorted(glob.glob("outputs/eval/*/eval_info.json")):
    with open(f) as fh:
        all_results[f] = json.load(fh)

with open("eval_comparison_v2.json", "w") as fh:
    json.dump(all_results, fh, indent=2)
print(f"Saved {len(all_results)} eval results to eval_comparison_v2.json")

import os as _os
_src = _os.path.abspath("eval_comparison_v2.json")
_dst = "/kaggle/working/eval_comparison_v2.json"
if _src != _dst:
    shutil.copy(_src, _dst)
    print(f"Artifact copied to {_dst}")
else:
    print(f"Artifact already at {_dst} (same path)")